In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import tqdm
import functools
import sys

import pandas as pd

import matplotlib.pyplot as plt

import numpy as np

import tensorflow as tf
import tensorflow_probability as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

In [ ]:
from discovery import collect_vep, load_panel, PROTEIN_AFFECTING_SET

In [ ]:
# path with mutations
base_folder = "/data/bbg/nobackup/prominent/chip"
deepCSA_run = '2026-04-14'
deepCSA_folder = f'{base_folder}/deepCSA/runs/{deepCSA_run}_deepCSA/{deepCSA_run}_CH_I_wSP_filtered_muts'
sample_group = 'CohortCha_TimepointT0'
impact = 'protein_affecting'
output_folder = f'{base_folder}/analysis/{deepCSA_run}_deepCSA_CH_I_wSP/{deepCSA_run}_CH_I_wSP_custom_analysis/saturation_mutagenesis/saturation_kinetics/plots/{sample_group}/'
os.makedirs(output_folder, exist_ok=True)

# load mutations

In [ ]:
mutations_genomic = pd.read_csv(f'{base_folder}/analysis/{deepCSA_run}_deepCSA_CH_I_wSP/{deepCSA_run}_CH_I_wSP_custom_analysis/saturation_mutagenesis/saturation_kinetics/{sample_group}_mutations_genomic_rates.{impact}.tsv', sep='\t')
mutations_residue = pd.read_csv(f'{base_folder}/analysis/{deepCSA_run}_deepCSA_CH_I_wSP/{deepCSA_run}_CH_I_wSP_custom_analysis/saturation_mutagenesis/saturation_kinetics/{sample_group}_mutations_residue_rates.{impact}.tsv', sep='\t')

In [ ]:
mutations_genomic.head()

In [ ]:
mutations_residue.head()

# compute empirical discovery index curves per gene

In [ ]:
vep = collect_vep(deepCSA_folder)
df_panel = load_panel(deepCSA_folder, sample_group, vep)


In [ ]:
# df_panel represents the total number of mutable sites,
# either genomic or residue sites
print(df_panel.shape)
if impact == "protein_affecting":
    df_panel = df_panel[df_panel["IMPACT"].isin(PROTEIN_AFFECTING_SET)]
else:
    df_panel = df_panel[df_panel["IMPACT"] == "synonymous"]
    
print(df_panel.shape)

In [ ]:
df_panel_genomic = df_panel.groupby(['POS', 'GENE']).agg({'DEPTH': 'mean'}).reset_index()
df_panel_residue = df_panel.groupby(['RESIDUE',  'GENE']).agg({'DEPTH': 'mean'}).reset_index()

In [ ]:
df_panel_dict = {
    'genomic': df_panel_genomic,
    'residue': df_panel_residue
    }

mutations_dict = {
    'genomic': mutations_genomic,
    'residue': mutations_residue
}

In [ ]:
subsampling_rates = np.logspace(-2, np.log10(0.9), num=20)

def empirical_discovery_index_curve(gene, replicates=100, sites='genomic'):

    df = mutations_dict[sites]
    df = df[df['GENE'] == gene]

    dg = df_panel_dict[sites]
    dg = dg[dg['GENE'] == gene]

    size = dg.shape[0]
    mean_depth = df['DEPTH'].mean()
    
    x, mean, err_low, err_high = [], [], [], []
    
    unique_dict = {}
    for i, p in tqdm.tqdm(enumerate(subsampling_rates)):
        dist_bernoulli = tfd.Bernoulli(probs=df[f'UNIQUE_RATE_{i}'].values)
        unique_mutations = np.sum(dist_bernoulli.sample(sample_shape=(100,)), axis=1)
        y = list(unique_mutations / size)
        mean += [np.mean(y)]
        err_low += [np.percentile(y, 2.5)]
        err_high += [np.percentile(y, 97.5)]
        x += [mean_depth * p]
    mean += [df.shape[0] / size]
    err_low += [df.shape[0] / size]
    err_high += [df.shape[0] / size]
    x += [mean_depth]

    return x, mean, err_low, err_high

In [ ]:
def plot_empirical_discovery(gene):
    x, mean, err_low, err_high = empirical_discovery_index_curve(gene, sites='genomic')
    plt.scatter(x, mean, s=50)
    for i, m in enumerate(x):
        plt.vlines(m, err_low[i], err_high[i])
    plt.xscale('log')
    plt.title(gene)
    plt.show()

In [ ]:
plot_empirical_discovery('DNMT3A')

# theoretical neutral vs empirical discovery curves

In [ ]:
# Create a PDF to save the plots
def main_empirical(sample, sites='genomic', impact = "protein_affecting", logscale=False, genes_list = None):

    mutations_lite = mutations_dict[sites]
    mutations_lite['VAF'] = mutations_lite.apply(lambda r: r['ALT_DEPTH']/r['DEPTH'], axis=1)

    df_panel = pd.read_csv(os.path.join(deepCSA_folder, 'regions', 'consensuspanels', 'consensus.exons_splice_sites.tsv'), sep='\t')

    # include depth per site

    df_depth = pd.read_csv(os.path.join(deepCSA_folder, 'depths', 'individual',f'{sample}.depths.annotated.tsv.gz'), sep='\t')
    df_panel = pd.merge(df_panel, df_depth[['CHROM', 'POS', sample]], on=['CHROM', 'POS'], how='left')
    df_panel.rename(columns={sample: 'DEPTH'}, inplace=True)

    # retrieve relative mutability
    mutability_raw = pd.read_csv(os.path.join(deepCSA_folder, 'processing_files', 'absolutemutabilities', f'mutabilities_per_site.{sample}.tsv.gz'), 
                                    sep='\t')

    mutability_raw = pd.merge(mutability_raw, df_panel, on=['CHROM', 'POS', 'CONTEXT_MUT', 'GENE','IMPACT'], how='left')
    mutability_raw = mutability_raw.rename(columns={sample: "MUTABILITY"})
    # collect VEP annotations

    df_panel = pd.merge(df_panel, vep[['CHROM', 'POS', 'REF', 'ALT', 'AACHANGE', 'GENE']], 
                              left_on=['CHROM', 'POS', 'REF', 'ALT', 'GENE'], 
                              right_on=['CHROM', 'POS', 'REF', 'ALT', 'GENE'],
                              how='left')

    if genes_list is None:
        genes_list = df_panel['GENE'].unique()

    for gene in tqdm.tqdm(genes_list):

        try:
            synonymous_mutation_rate = pd.read_csv(os.path.join(deepCSA_folder, 'selection','omega', 'preprocessing', f'mutability_per_sample_gene_context.{sample}.tsv'), sep='\t')
            synonymous_mutation_rate = synonymous_mutation_rate[synonymous_mutation_rate['GENE'] == gene]
            mutability_gene = mutability_raw[mutability_raw['GENE'] == gene]
            mutability_gene = pd.merge(mutability_gene, synonymous_mutation_rate[['CONTEXT_MUT', sample]], on=['CONTEXT_MUT'], how='left')
            mutability_gene.rename(columns={sample: 'MUTRATE'}, inplace=True)

            mutability_gene = pd.merge(mutability_gene, df_panel[['CHROM', 'POS', 'REF', 'ALT', 'AACHANGE']], on=['CHROM', 'POS', 'REF', 'ALT'], how='left')

            # discard positions in non-CDS regions, probably splicing and intronic
            mutability_gene = mutability_gene[(mutability_gene['AACHANGE'] != '-') & (~mutability_gene['AACHANGE'].isnull())]

            # keep only protein affecting mutation sites
            if impact == "protein_affecting":
                mutability_gene = mutability_gene[mutability_gene["IMPACT"].isin(PROTEIN_AFFECTING_SET)]
            else:
                mutability_gene = mutability_gene[mutability_gene["IMPACT"] == 'synonymous']

            mutability_gene['RESIDUE'] = mutability_gene['AACHANGE'].apply(lambda s: s[:-1])
            print(mutability_gene.head())
            if sites == 'residue':
                mutability_gene = mutability_gene.groupby(['GENE', 'RESIDUE']).agg({'MUTRATE': 'sum', 'DEPTH': 'mean'}).reset_index()
            elif sites == 'genomic':
                mutability_gene = mutability_gene.groupby(['GENE', 'POS']).agg({'MUTRATE': 'sum', 'DEPTH': 'mean'}).reset_index()
            
            mutability_gene['MUTABILITY'] = mutability_gene.apply(lambda s: (s['MUTRATE'] / s['DEPTH']), axis=1)

            if sites == 'residue':
                mutability_gene = pd.merge(mutability_gene, mutations_lite[['GENE', 'RESIDUE', 'VAF']], on=['GENE', 'RESIDUE'], how='left')
            elif sites == 'genomic':
                mutability_gene = pd.merge(mutability_gene, mutations_lite[['GENE', 'POS', 'VAF']], on=['GENE', 'POS'], how='left')


            # neutral rate
            mutability_gene['RATE_NEUTRAL'] = mutability_gene['MUTABILITY']
            total_neutral_rate = mutability_gene['RATE_NEUTRAL'].sum()

            # compute saturation theoretical
            y_unique_neutral = []
            if sites == 'genomic':
                x_theoretical = np.logspace(3, 8, num=100)
            elif sites == 'residue':
                x_theoretical = np.logspace(3, 7, num=100)

            for depth in x_theoretical:
                unique_neutral = np.sum(1 - np.exp(-mutability_gene['RATE_NEUTRAL'].values * depth)) / mutability_gene.shape[0]
                y_unique_neutral.append(unique_neutral)

            # compute empirical discovery index curve

            x_empirical, mean, err_low, err_high = empirical_discovery_index_curve(gene, sites=sites)
            # plot

            fig, ax1 = plt.subplots(figsize=(2,2))
            ax1.set_xscale('log')
            if logscale:
                ax1.set_yscale('log')

            # empirical

            ax1.scatter(x_empirical[:-1], mean[:-1], label='downsampling', color='brown', s=5)
            ax1.scatter(x_empirical[-1], mean[-1], label='observed', color='white', edgecolors='brown', alpha=1, s=100)
            for i, m in enumerate(x_empirical[:-2]):
                plt.vlines(m, err_low[i], err_high[i], color='brown', lw=1)

            # theoretical

            ax1.plot(x_theoretical, y_unique_neutral, color='grey', lw=2, label='neutral theoretical', alpha=0.5)  # neutral
            # ax1.tick_params(axis='y', labelcolor=color)
            if sites == 'residue':
                ax1.set_ylabel('proportion of\nmutated residues')
            elif sites == 'genomic':
                ax1.set_ylabel('proportion of\nmutated nucleotides')
            
            ax1.set_xlabel('depth per residue')
            # ax1.vlines(5e5, 0, 1., linestyles='dashed', color='maroon', label='cohort', alpha=0.3)

            ax1.spines['top'].set_visible(False)
            ax1.spines['right'].set_visible(False)

            ax1.set_xlim(x_theoretical[0], x_theoretical[-1])

            # ax1.legend(loc=(1,0))

            plt.title(gene + " (" + impact + ")")

            if logscale:
                plt.savefig(f'{output_folder}/proportion_mutated_sites_{sites}_logscale_{gene}.{impact}.png', bbox_inches='tight', dpi=300)
                print(f"Plot saved to {output_folder}")
            else:
                plt.savefig(f'{output_folder}/proportion_mutated_sites_{sites}_{gene}.{impact}.png', bbox_inches='tight', dpi=300)
                print(f"Plot saved to {output_folder}")

            plt.show()

        except:

            print(gene)
            continue

In [ ]:
main_empirical('CohortCha_TimepointT0', sites='genomic', impact = impact, logscale=False, genes_list = ["DNMT3A","TET2","PPM1D","TP53","CHEK2","ASXL1"])

In [ ]:
main_empirical('CohortCha_TimepointT0', sites='residue', impact=impact, logscale=False,genes_list = ["DNMT3A","TET2","PPM1D","TP53","CHEK2","ASXL1"])